# C3-01 · Label-free recalibration (C3) on the development events (CPU)

Estimates each development event's share of damaged buildings **without its labels**, from the probes'
held-out scores (p03 run), and checks the C3 gate. Rule fixed in `recal/prevalence.py` and the pre-registration
before any C3 result was computed:

* Platt calibration on the source (the other events' held-out scores), EM for the target prior (reported),
  BBSE as a cross-check; **abstain** if the source tpr − fpr at 0.5 is below 0.2 or EM and BBSE disagree by
  more than 50%.
* Baseline: the training-average prevalence.
* **C3 gate:** relative error ≤ 50% on ≥ 4 of 5 events and better than the baseline on ≥ 4 of 5
  (abstentions count as misses). Gated on DINOv3 outline; ImageNet outline reported too.

* **Runtime:** CPU is enough (Runtime → Change runtime type → CPU). Secret: `GH_TOKEN`. Needs the p03 run.
* **Time:** about 5 minutes. **Outputs:** `c3_estimates.csv`, `c3_gate.json` in this run's folder.

In [ ]:
NOTEBOOK_NAME = "c301_c3_recalibration"

# Drive layout (FYP Google account). Keep in sync with src/damagesar_bench/drive.py
import os, datetime, re
from google.colab import drive
drive.mount("/content/drive")
PROJECT = "/content/drive/MyDrive/DamageSAR-Bench"
FOLDERS = ["00_admin", "01_data/raw/bright", "01_data/raw/instance_labels", "01_data/external",
           "01_data/derived", "02_models/checkpoints", "02_models/features", "03_notebooks/pipeline",
           "03_notebooks/checks", "03_notebooks/scratch", "04_outputs/runs", "04_outputs/reports"]
for f in FOLDERS:
    os.makedirs(f"{PROJECT}/{f}", exist_ok=True)
def run_dir(name):
    slug = re.sub(r"[^a-z0-9]+", "-", name.lower()).strip("-")
    p = f"{PROJECT}/04_outputs/runs/{datetime.date.today().isoformat()}_{slug}"
    os.makedirs(p, exist_ok=True)
    return p
RUN = run_dir(NOTEBOOK_NAME)
print("Drive layout ready under", PROJECT)
print("This run saves to:", RUN)

In [ ]:
# Install this repo's package (private repo: needs GH_TOKEN in Colab Secrets, read access to damagesar-bench)
from google.colab import userdata
GH = userdata.get("GH_TOKEN")
!git clone -q https://{GH}@github.com/ThinukaLehan/damagesar-bench.git /content/damagesar-bench 2>/dev/null || (cd /content/damagesar-bench && git pull -q)
!pip install -q -e /content/damagesar-bench
COMMIT = !git -C /content/damagesar-bench rev-parse --short HEAD
COMMIT = COMMIT[0]
# A running Colab kernel does not see an editable install until a restart: put src/ on the path.
import sys, importlib
if "/content/damagesar-bench/src" not in sys.path:
    sys.path.insert(0, "/content/damagesar-bench/src")
importlib.invalidate_caches()
import damagesar_bench
print("damagesar-bench at commit", COMMIT, "->", damagesar_bench.__file__)

In [ ]:
# C3 on the p03 held-out scores
import glob, json
import pandas as pd
from damagesar_bench.recal.prevalence import c3_gate, estimate_all
P03 = sorted(glob.glob(f"{PROJECT}/04_outputs/runs/*_p03-controls-and-imagenet-v0/bench_run/fold_scores.parquet"))
assert P03, "missing the p03 run: run p03 first"
S = pd.read_parquet(P03[-1])
est = pd.concat([estimate_all(S, "dinov3_vitb16@native", "outline"),
                 estimate_all(S, "imagenet_vitb16@native", "outline")], ignore_index=True)
est.to_csv(f"{RUN}/c3_estimates.csv", index=False)
cols = ["encoder", "event", "n", "true_prevalence", "baseline", "em", "bbse", "youden", "abstain", "estimate",
        "count_estimate", "rel_error", "baseline_rel_error", "reason"]
display(est[cols].round(3))
gates = {}
for enc, g in est.groupby("encoder"):
    r = c3_gate(g)
    gates[enc] = {"n_events": r.n_events, "n_within_50pct": r.n_within, "n_beat_baseline": r.n_beat_baseline,
                  "n_abstain": r.n_abstain, "passed": r.passed}
json.dump(gates, open(f"{RUN}/c3_gate.json", "w"), indent=2)
print(json.dumps(gates, indent=2))
print("C3 GATE (DINOv3 outline):", "PASSED" if gates["dinov3_vitb16@native"]["passed"] else "NOT PASSED")

In [ ]:
# File the executed notebook (with outputs) into this run's folder and 03_notebooks/pipeline/
import json
from google.colab import _message
# Ask the browser for the notebook; it can time out (busy tab, large outputs). Results above are already saved.
nb = None
for attempt in range(3):
    try:
        reply = _message.blocking_request("get_ipynb", timeout_sec=120)
        nb = (reply or {}).get("ipynb")
    except Exception as ex:
        print("attempt", attempt + 1, "failed:", ex)
    if nb:
        break
if not nb:
    print("Could not copy the executed notebook automatically (the results above are already saved).")
    print("Use File -> Save a copy in Drive, then move the copy into", RUN)
else:
    stamp = datetime.date.today().isoformat()
    for target in (f"{RUN}/{NOTEBOOK_NAME}_executed.ipynb",
                   f"{PROJECT}/03_notebooks/pipeline/{stamp}_{NOTEBOOK_NAME}.ipynb"):
        with open(target, "w") as fh:
            json.dump(nb, fh)
        print("saved", target)
    print("run folder:", sorted(os.listdir(RUN)))